In [1]:
## Step 1: Check GPU Availability

# To detect GPU availability and print the GPU name.
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # 0=all, 1=filter INFO, 2=filter WARNING, 3=ERROR only

import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')

if gpus:
    for gpu in gpus:
        details = tf.config.experimental.get_device_details(gpu)
        print(details.get("device_name", "Unknown GPU"))
else:
    print("No GPU detected")


NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [3]:
## Step 2: Set Global Policy for Mixed Precision Training

# To set the global policy to 'mixed_float16' for mixed precision training.
import tensorflow as tf
from tensorflow.keras import mixed_precision

mixed_precision.set_global_policy('mixed_float16')

print("Global policy:", mixed_precision.global_policy())


Global policy: <DTypePolicy "mixed_float16">


In [4]:
DATASET_DIR = "./dataset_processed"
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode="rgb"
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Classes:", class_names)
print("Num classes:", num_classes)

AUTOTUNE = tf.data.AUTOTUNE

Found 10000 files belonging to 10 classes.
Using 8000 files for training.


I0000 00:00:1771829763.030619   30402 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Found 10000 files belonging to 10 classes.
Using 2000 files for validation.
Classes: ['ක', 'ග', 'ත', 'න', 'ප', 'ම', 'ර', 'ල', 'ස', 'හ']
Num classes: 10


In [5]:
from tensorflow.keras.applications.convnext import preprocess_input

def preprocess_ds(images, labels):
    images = preprocess_input(images)
    return images, labels

train_ds = train_ds.map(preprocess_ds, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(preprocess_ds, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

In [6]:
from tensorflow.keras.applications import ConvNeXtBase

base_model = ConvNeXtBase(
    include_top=False,
    weights="imagenet",
    input_shape=(128, 128, 3)
)

# 🔥 NO freezing
base_model.trainable = True

inputs = layers.Input(shape=(128,128,3))
x = base_model(inputs, training=True)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation="relu")(x)
x = layers.Dropout(0.4)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax",
    dtype="float32"  # mixed precision safety
)(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ convnext_base (Functional)      │ (None, 4, 4, 1024)     │    87,566,464 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1024)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1024)           │         4,096 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       524,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 88,100,490 (336.08 MB)

 Trainable params: 88,098,442 (336.07 MB)

 Non-trainable params: 2,048 (8.00 KB)

In [7]:
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping

callbacks = [
    ModelCheckpoint("best_convnext_full.keras",
                    monitor="val_accuracy",
                    save_best_only=True),

    ReduceLROnPlateau(monitor="val_loss",
                      factor=0.5,
                      patience=3,
                      verbose=1),

    EarlyStopping(monitor="val_accuracy",
                  patience=6,
                  restore_best_weights=True,
                  verbose=1)
]

In [8]:
import time

start = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
    callbacks=callbacks
)

print("Training time (minutes):", (time.time() - start)/60)

Epoch 1/6


I0000 00:00:1771829848.757673   30691 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


250/250 ━━━━━━━━━━━━━━━━━━━━ 363s 750ms/step - accuracy: 0.9473 - loss: 0.1881 - val_accuracy: 0.9900 - val_loss: 0.0385 - learning_rate: 1.0000e-04
Epoch 2/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 119s 476ms/step - accuracy: 0.9859 - loss: 0.0446 - val_accuracy: 0.9905 - val_loss: 0.0242 - learning_rate: 1.0000e-04
Epoch 3/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 108s 431ms/step - accuracy: 0.9875 - loss: 0.0365 - val_accuracy: 0.9905 - val_loss: 0.0241 - learning_rate: 1.0000e-04
Epoch 4/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 108s 431ms/step - accuracy: 0.9874 - loss: 0.0349 - val_accuracy: 0.9865 - val_loss: 0.0252 - learning_rate: 1.0000e-04
Epoch 5/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 109s 435ms/step - accuracy: 0.9875 - loss: 0.0326 - val_accuracy: 0.9905 - val_loss: 0.0224 - learning_rate: 1.0000e-04
Epoch 6/6
250/250 ━━━━━━━━━━━━━━━━━━━━ 109s 435ms/step - accuracy: 0.9875 - loss: 0.0297 - val_accuracy: 0.9905 - val_loss: 0.0200 - learning_rate: 1.0000e-04
Restoring model weights from the end of the best epoch: 

In [9]:
model.save("sinhala_convnext_full.keras")
print("Model saved successfully.")

Model saved successfully.
